# 02 Vector Database

本章目标：

1. 理解为什么需要向量数据库
2. 理解 Exact Search 与 Approximate Nearest Neighbor
3. 理解 ANN 与 HNSW 的基本思想
4. 掌握向量数据库中的 Collection、Point、Payload、Index
5. 使用 Qdrant 创建向量集合
6. 完成向量写入、检索和过滤
7. 构建一个持久化的小型知识库

## 1. Exact Search、ANN 与 HNSW

本节目标：

1. 理解 Exact Search
2. 理解 Approximate Nearest Neighbor（ANN）
3. 理解 Recall 与 Search Speed 的权衡
4. 理解 HNSW 的基本思想
5. 明白为什么向量数据库需要索引

### Exact Search vs HNSW

| 特性   | Exact Search | HNSW |
|---    |---|---|
| 是否扫描所有向量 | 是 | 否 |
| 结果 | 精确 | 近似 |
| 查询速度 | 数据量大时慢 | 快 |
| 是否需要索引 | 不需要 | 需要 |
| 建库成本 | 低 | 较高 |
| 内存 | 相对低 | 较高 |
| 适合场景 | 小规模数据 | 大规模向量检索 |

In [2]:
import numpy as np
import time
def exact_search(query, vectors, top_k=5):
    scores = vectors @ query # 计算相似度

    top_indices = np.argsort(scores)[::-1][:top_k]

    return top_indices, scores[top_indices]

dimension = 512

query = np.random.randn(dimension).astype(np.float32)
query = query / np.linalg.norm(query)
sizes = [
    1_000,
    10_000,
    100_000
]
for size in sizes:

    vectors = np.random.randn(
        size,
        dimension
    ).astype(np.float32)

    vectors /= np.linalg.norm(
        vectors,
        axis=1,
        keepdims=True
    )

    start = time.perf_counter()

    indices, scores = exact_search(
        query,
        vectors,
        top_k=5
    )

    elapsed = time.perf_counter() - start

    print(
        f"N={size:,} | "
        f"time={elapsed:.6f}s"
    )

N=1,000 | time=0.000277s
N=10,000 | time=0.000785s
N=100,000 | time=0.008392s


## 2. Vector Database 核心概念

本节目标：

1. 理解 Collection
2. 理解 Point
3. 理解 Vector
4. 理解 Payload / Metadata
5. 理解 Filter
6. 理解 Index
7. 建立 Qdrant 中这些概念之间的对应关系

In [3]:
point_example = {
    "id": 1,

    "vector": [
        0.12,
        -0.31,
        0.52
    ],

    "payload": {
        "text": "持续降雨会提高孔隙水压力",
        "source": "landslide_report.pdf",
        "page": 12,
        "year": 2026,
        "category": "landslide"
    }
}

point_example

{'id': 1,
 'vector': [0.12, -0.31, 0.52],
 'payload': {'text': '持续降雨会提高孔隙水压力',
  'source': 'landslide_report.pdf',
  'page': 12,
  'year': 2026,
  'category': 'landslide'}}

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)
print("Qdrant imports OK")

client = QdrantClient(
    path="./qdrant_data"
)


Qdrant imports OK


In [5]:
from sentence_transformers import SentenceTransformer
import tqdm as notebook_tqdm
from rich import print as rprint
model_name = "BAAI/bge-small-zh-v1.5"
model = SentenceTransformer(model_name)

collection_name = "learning_knowledge"
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=512,
            distance=Distance.COSINE
        )
    )
client.get_collections()
# client.close()
documents = [
    "持续降雨会导致土体含水率增加，并可能提高孔隙水压力。",
    "坡度是影响滑坡稳定性的重要地形因素。",
    "Python广泛用于数据分析和机器学习。",
    "太阳辐射是影响光伏发电功率的重要气象因素。",
    "高温天气可能降低光伏组件的转换效率。",
    "地下水位升高可能降低边坡稳定性。"
]

# embedding
document_embeddings = model.encode(
    documents,
    normalize_embeddings=True
)

points = []

for i, (text, vector) in enumerate(
    zip(documents, document_embeddings)
):
    point = PointStruct(
        id=i,
        vector=vector.tolist(),
        payload={
            "text": text
        }
    )

    points.append(point)
# rprint(points[0])
client.upsert(
    collection_name=collection_name,
    points=points
)
collection_info = client.get_collection(collection_name)
# rprint(collection_info)

c:\Users\Administrator\miniconda3\envs\agent_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 71/71 [00:00<00:00, 17756.58it/s]


In [6]:
query = "连续降雨为什么会增加滑坡风险？"

query_vector = model.encode(query,normalize_embeddings=True)

result = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    limit=3,

)
# result
for point in result.points:
    print(
        f"score={point.score:.4f}"
    )

    print(
        point.payload["text"]
    )
    
    print("-" * 60)

score=0.7072
持续降雨会导致土体含水率增加，并可能提高孔隙水压力。
------------------------------------------------------------
score=0.6927
坡度是影响滑坡稳定性的重要地形因素。
------------------------------------------------------------
score=0.6926
地下水位升高可能降低边坡稳定性。
------------------------------------------------------------


## 3. Payload Metadata 与 Filter

本节目标：

1. 理解 Payload 的作用
2. 给向量增加结构化元数据
3. 使用 Filter 进行条件检索
4. 理解“语义检索 + 结构化过滤”的组合
5. 为后续 RAG 中的来源过滤、年份过滤、类别过滤做准备

In [7]:
documents = [
    {
        "text": "持续降雨会导致土体含水率增加，并可能提高孔隙水压力。",
        "category": "landslide",
        "source": "landslide_report.pdf",
        "year": 2026,
        "page": 12
    },
    {
        "text": "坡度是影响滑坡稳定性的重要地形因素。",
        "category": "landslide",
        "source": "landslide_report.pdf",
        "year": 2026,
        "page": 15
    },
    {
        "text": "Python广泛用于数据分析和机器学习。",
        "category": "programming",
        "source": "python_notes.md",
        "year": 2025,
        "page": 1
    },
    {
        "text": "太阳辐射是影响光伏发电功率的重要气象因素。",
        "category": "photovoltaic",
        "source": "pv_report.pdf",
        "year": 2026,
        "page": 8
    },
    {
        "text": "高温天气可能降低光伏组件的转换效率。",
        "category": "photovoltaic",
        "source": "pv_report.pdf",
        "year": 2024,
        "page": 11
    },
    {
        "text": "地下水位升高可能降低边坡稳定性。",
        "category": "landslide",
        "source": "groundwater_report.pdf",
        "year": 2025,
        "page": 6
    }
]
# 提取文本
texts = [
    item["text"]
    for item in documents
]
# 重做embedding
document_embeddings = model.encode(
    texts,
    normalize_embeddings=True
)

# 重新构建points
points = []

for i, (item, vector) in enumerate(
    zip(documents, document_embeddings)
):
    point = PointStruct(
        id=i,
        vector=vector.tolist(),
        payload=item
    )

    points.append(point)

client.upsert(collection_name=collection_name, points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [8]:
query = "什么因素会导致边坡失稳？"

query_vector = model.encode(
    query,
    normalize_embeddings=True
)

result = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    limit=5
)
for point in result.points:
    print(f"score={point.score:.4f}")
    print("text:", point.payload["text"])
    print("category:", point.payload["category"])
    print("year:", point.payload["year"])
    print("source:", point.payload["source"])
    print("-" * 60)

score=0.7778
text: 坡度是影响滑坡稳定性的重要地形因素。
category: landslide
year: 2026
source: landslide_report.pdf
------------------------------------------------------------
score=0.7344
text: 地下水位升高可能降低边坡稳定性。
category: landslide
year: 2025
source: groundwater_report.pdf
------------------------------------------------------------
score=0.5459
text: 持续降雨会导致土体含水率增加，并可能提高孔隙水压力。
category: landslide
year: 2026
source: landslide_report.pdf
------------------------------------------------------------
score=0.4035
text: 高温天气可能降低光伏组件的转换效率。
category: photovoltaic
year: 2024
source: pv_report.pdf
------------------------------------------------------------
score=0.3954
text: 太阳辐射是影响光伏发电功率的重要气象因素。
category: photovoltaic
year: 2026
source: pv_report.pdf
------------------------------------------------------------


In [9]:
from qdrant_client.models import (
    Filter,
    FieldCondition,
    MatchValue
)

landslide_filter = Filter(
    must=[
        FieldCondition(
            key="category",
            match=MatchValue(value="landslide")
        )
    ]
)
result = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    query_filter=landslide_filter,
    limit=5
)
for point in result.points:
    print(f"score={point.score:.4f}")
    print(point.payload)
    print("-" * 60)

score=0.7778
{'text': '坡度是影响滑坡稳定性的重要地形因素。', 'category': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 15}
------------------------------------------------------------
score=0.7344
{'text': '地下水位升高可能降低边坡稳定性。', 'category': 'landslide', 'source': 'groundwater_report.pdf', 'year': 2025, 'page': 6}
------------------------------------------------------------
score=0.5459
{'text': '持续降雨会导致土体含水率增加，并可能提高孔隙水压力。', 'category': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12}
------------------------------------------------------------


In [10]:
# 过滤条件为 photovoltaic
pv_filter = Filter(
    must = [
        FieldCondition(
            key="category",
            match=MatchValue(value="photovoltaic")
        )
    ]
)

query = "天气条件如何影响能源输出？"

query_vector = model.encode(
    query,
    normalize_embeddings=True
)
result = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    query_filter=pv_filter,
    limit=5
)
for point in result.points:
    print(f"score={point.score:.4f}")
    print(point.payload)
    


score=0.6792
{'text': '太阳辐射是影响光伏发电功率的重要气象因素。', 'category': 'photovoltaic', 'source': 'pv_report.pdf', 'year': 2026, 'page': 8}
score=0.6597
{'text': '高温天气可能降低光伏组件的转换效率。', 'category': 'photovoltaic', 'source': 'pv_report.pdf', 'year': 2024, 'page': 11}


In [11]:
# 过滤条件为时间
from qdrant_client.models import Range

year_filter = Filter(
    must=[
        FieldCondition(
            key="year",
            range=Range(
                gte=2025
                
            )
        )
    ]
)
query = "光伏功率受到什么因素影响？"

query_vector = model.encode(
    query,
    normalize_embeddings=True
)

result = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    query_filter=year_filter,
    limit=5
)
for point in result.points:
    print(f"score={point.score:.4f}")
    print(point.payload)

score=0.7889
{'text': '太阳辐射是影响光伏发电功率的重要气象因素。', 'category': 'photovoltaic', 'source': 'pv_report.pdf', 'year': 2026, 'page': 8}
score=0.5658
{'text': '持续降雨会导致土体含水率增加，并可能提高孔隙水压力。', 'category': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12}
score=0.5003
{'text': '地下水位升高可能降低边坡稳定性。', 'category': 'landslide', 'source': 'groundwater_report.pdf', 'year': 2025, 'page': 6}
score=0.4913
{'text': '坡度是影响滑坡稳定性的重要地形因素。', 'category': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 15}
score=0.3399
{'text': 'Python广泛用于数据分析和机器学习。', 'category': 'programming', 'source': 'python_notes.md', 'year': 2025, 'page': 1}


## 4. Qdrant 持久化与数据库管理

本节目标：

1. 理解 Local Mode 的持久化机制
2. 掌握 Collection 的查看、删除与重建
3. 掌握 Point 的更新与删除
4. 理解 Payload Index
5. 理解 Local Mode 与 Server Mode 的区别
6. 为后续 Docker 部署 Qdrant 做准备

In [12]:
info = client.get_collection(
    collection_name="learning_knowledge"
)

rprint(info)

CollectionInfo(
    status=<CollectionStatus.GREEN: 'green'>,
    optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>,
    warnings=None,
    indexed_vectors_count=0,
    points_count=6,
    segments_count=1,
    config=CollectionConfig(
        params=CollectionParams(
            vectors=VectorParams(
                size=512,
                distance=<Distance.COSINE: 'Cosine'>,
                hnsw_config=None,
                quantization_config=None,
                on_disk=None,
                memory=None,
                datatype=None,
                multivector_config=None
            ),
            shard_number=None,
            sharding_method=None,
            replication_factor=None,
            write_consistency_factor=None,
            read_fan_out_factor=None,
            read_fan_out_delay_ms=None,
            on_disk_payload=None,
            payload=None,
            id_tracker=None,
            sparse_vectors=None
        ),
        hnsw_config=HnswConfig(
            m=16,
            ef_construct=100,
            full_scan_threshold=10000,
            max_indexing_threads=0,
            on_disk=None,
            memory=None,
            payload_m=None,
            inline_storage=None
        ),
        optimizer_config=OptimizersConfig(
            deleted_threshold=0.2,
            vacuum_min_vector_number=1000,
            default_segment_number=0,
            max_segment_size=None,
            memmap_threshold=None,
            indexing_threshold=20000,
            flush_interval_sec=5,
            max_optimization_threads=1,
            prevent_unoptimized=None
        ),
        wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_retain_closed=1),
        quantization_config=None,
        strict_mode_config=None,
        metadata=None
    ),
    payload_schema={},
    update_queue=None,
    created_at=None
)

In [13]:
collection_name = "learning_knowledge"

exists = client.collection_exists(
    collection_name=collection_name
)

print(exists)

if not client.collection_exists(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        vector_size=512,
        distance="Cosine",
    )

True


In [14]:
from qdrant_client.models import (PointStruct, VectorParams, Distance) #  导入Qdrant客户端模型中的必要组件

print(points[0].payload) #  打印points列表中第一个点的payload信息

new_text = "持续降雨会导致土体含水率增加，并提高孔隙水压力。" #  定义新的文本内容
new_vector = model.encode(new_text , normalize_embeddings=True) #  使用模型对文本进行编码，并归一化嵌入向量

update_point = PointStruct( #  创建新的点结构，用于更新向量数据库中的数据
    id=0, #  点的ID，用于标识唯一向量
    vector=new_vector.tolist(), #  将编码后的向量转换为列表格式
    payload={ #  载荷信息，包含与向量相关的元数据
        "text": new_text, #  原始文本内容
        "catergory": "landslide", #  数据类别（注意：此处"catergory"应为"category"）
        "source":"landslide_report.pdf", #  数据来源文件
        "year" :2026, #  数据年份
        "page": 12, #  来源文件的页码
        "region": "Kentucky" #  相关地区

    }
)

client.upsert(collection_name=collection_name, points=[update_point])    #  将新的点数据更新到向量数据库中

result = client.retrieve(collection_name=collection_name, ids=[0]) #  查询ID为0的点数据

print(result) #  再次打印更新后points列表中第一个点的payload信息

{'text': '持续降雨会导致土体含水率增加，并可能提高孔隙水压力。', 'category': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12}
[Record(id=0, payload={'text': '持续降雨会导致土体含水率增加，并提高孔隙水压力。', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky'}, vector=None, shard_key=None, order_value=None)]


In [15]:
from qdrant_client.models import PayloadSchemaType
client.create_payload_index(
    collection_name = collection_name,
    field_name = "year",
    field_schema= PayloadSchemaType.INTEGER
)
client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=PayloadSchemaType.KEYWORD
)
client.close()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_26284\1269512245.py:2: UserWarning: Payload indexes have no effect in the local Qdrant, except text indexes with `scoring`, which enable text queries. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


## 5. Docker Server Mode

本节目标：

1. 理解 Local Mode 与 Server Mode 的区别
2. 使用 Docker 启动独立 Qdrant 服务
3. 理解端口映射
4. 理解 Docker Volume
5. 使用浏览器访问 Qdrant Dashboard
6. 使用 Python Client 连接 Qdrant Server
7. 将本地实验迁移到真正的服务端数据库

## 6. Docker 基础：Image、Container、Port、Volume

本节目标：

1. 理解 Docker Image
2. 理解 Docker Container
3. 理解端口映射
4. 理解 Volume 持久化
5. 使用 Docker 启动 Qdrant
6. 使用 Python Client 连接独立 Qdrant Server

In [17]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url="http://localhost:6333",
    grpc_port=6334
)


client.get_collections()

CollectionsResponse(collections=[])

In [19]:
from qdrant_client.models import PointStruct, VectorParams, Distance

collection_name = "learning_knowledge"

if not client.collection_exists(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=512, distance=Distance.COSINE),
    )
client.get_collections()

client.upsert(
    collection_name=collection_name,
    points =  points
)

info = client.get_collection(collection_name=collection_name)
rprint(info.points_count)

6

In [20]:
query = "天气条件如何影响能源输出？"

query_vector = model.encode(query, normalize_embeddings=True)

result = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    limit=5,
    with_payload=True,
)

for point in result.points:
    print(f"score={point.score:.4f}")
    print(point.payload)

score=0.6792
{'text': '太阳辐射是影响光伏发电功率的重要气象因素。', 'category': 'photovoltaic', 'source': 'pv_report.pdf', 'year': 2026, 'page': 8}
score=0.6597
{'text': '高温天气可能降低光伏组件的转换效率。', 'category': 'photovoltaic', 'source': 'pv_report.pdf', 'year': 2024, 'page': 11}
score=0.5526
{'text': '持续降雨会导致土体含水率增加，并可能提高孔隙水压力。', 'category': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12}
score=0.4791
{'text': '坡度是影响滑坡稳定性的重要地形因素。', 'category': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 15}
score=0.4786
{'text': '地下水位升高可能降低边坡稳定性。', 'category': 'landslide', 'source': 'groundwater_report.pdf', 'year': 2025, 'page': 6}


## 7. 从 Document 到 Chunk：构建真正的知识库导入流程

本节目标：

1. 理解为什么不能“一篇文档 = 一个向量”
2. 理解 Document 与 Chunk 的区别
3. 学会基础文本切分
4. 为每个 Chunk 添加 Metadata
5. 批量生成 Embedding
6. Batch Upsert 到 Qdrant
7. 构建可追溯来源的知识库

In [21]:
document_text = """
持续降雨是诱发滑坡的重要因素之一。降雨入渗会导致土体含水率增加，
并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。

地下水位变化同样会影响边坡稳定性。当地下水位升高时，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，
但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。

在光伏功率预测中，太阳辐射是最重要的气象因素之一。
云量、温度、湿度和风速都会影响光伏组件的发电效率。
"""
paragraphs = [
    p.strip()
    for p in document_text.split("\n\n")
    if p.strip()
] 
print(paragraphs)

print("chunk数量: ", len(paragraphs))
for i, chunk in enumerate(paragraphs):
    print(f"\n chunk {i}")
    print(chunk)



['持续降雨是诱发滑坡的重要因素之一。降雨入渗会导致土体含水率增加，\n并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。', '地下水位变化同样会影响边坡稳定性。当地下水位升高时，\n坡体内部孔隙水压力可能增加，从而减小有效应力，\n使边坡更容易发生失稳。', '坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，\n但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。', '在光伏功率预测中，太阳辐射是最重要的气象因素之一。\n云量、温度、湿度和风速都会影响光伏组件的发电效率。']
chunk数量:  4

 chunk 0
持续降雨是诱发滑坡的重要因素之一。降雨入渗会导致土体含水率增加，
并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。

 chunk 1
地下水位变化同样会影响边坡稳定性。当地下水位升高时，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

 chunk 2
坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，
但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。

 chunk 3
在光伏功率预测中，太阳辐射是最重要的气象因素之一。
云量、温度、湿度和风速都会影响光伏组件的发电效率。


In [30]:
def simple_chunk_text(
    text,
    chunk_size=100,
    overlap=20
):
    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks
chunks = simple_chunk_text(document_text,chunk_size=50, overlap=20)
print(f"chunks数量: {len(chunks)}")
for i,chunk in enumerate(chunks):
    print(f"\nchunk {i}")
    print(chunk)

chunks数量: 9

chunk 0

持续降雨是诱发滑坡的重要因素之一。降雨入渗会导致土体含水率增加，
并可能造成孔隙水压力升高，从而降

chunk 1
增加，
并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。

地下水位变化同样会影响边坡稳定

chunk 2
强度。

地下水位变化同样会影响边坡稳定性。当地下水位升高时，
坡体内部孔隙水压力可能增加，从而减小

chunk 3
，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

坡度是影响滑坡发生的重

chunk 4
容易发生失稳。

坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，
但滑坡发生还

chunk 5
度通常意味着更大的下滑力，
但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。

在

chunk 6
植被覆盖和水文条件等因素共同控制。

在光伏功率预测中，太阳辐射是最重要的气象因素之一。
云量、温度

chunk 7
辐射是最重要的气象因素之一。
云量、温度、湿度和风速都会影响光伏组件的发电效率。


chunk 8
光伏组件的发电效率。



In [31]:
#构建metadata
chunk_records = []
for i, chunk in enumerate(chunks):
    chunk_record = {
        "text": chunk,
        "catergory": "landslide",
        "source": "landslide_report.pdf",
        "year": 2026,
        "page": 12,
        "region": "Kentucky",
        "chunk_id": i,
        "document_id": "doc_001"
    }
    chunk_records.append(chunk_record)
chunk_records[0]

{'text': '\n持续降雨是诱发滑坡的重要因素之一。降雨入渗会导致土体含水率增加，\n并可能造成孔隙水压力升高，从而降',
 'catergory': 'landslide',
 'source': 'landslide_report.pdf',
 'year': 2026,
 'page': 12,
 'region': 'Kentucky',
 'chunk_id': 0,
 'document_id': 'doc_001'}

In [39]:
# 批量生成Embedding
chunk_texts = [record["text"] for record in chunk_records]
chunk_embedding = model.encode(chunk_texts, normalize_embeddings=True,show_progress_bar=True)
print(chunk_embedding[0])
# 批量Upsert到Qdrant
chunk_points = []
for i, (chunk_text, embedding) in enumerate(zip(chunk_records, chunk_embedding)):
    point = PointStruct(
        id=i, #  点的ID，用于标识唯一向量
        vector=embedding.tolist(), #  将编码后的向量转换为列表格式
        payload=chunk_text
    )
    chunk_points.append(point)
print(chunk_points[0]) 

Batches: 100%|██████████| 1/1 [00:00<00:00, 22.05it/s]

[-6.35399669e-02 -5.52469455e-02  8.15878157e-03  3.16407010e-02
 -4.02529910e-02 -3.46960835e-02 -3.64247784e-02  3.45176868e-02
 -2.67139310e-03  1.33698350e-02  6.10019490e-02 -2.99327552e-01
 -4.09676768e-02 -3.27801891e-02  3.48919928e-02  2.31167320e-02
 -1.62002593e-02  2.35530827e-03 -1.87242292e-02  1.18709011e-02
  2.86982320e-02 -1.45536717e-02  1.29051032e-02  1.73195247e-02
 -1.06928544e-02 -5.28637320e-03  1.32510653e-02 -8.24135542e-03
  6.43566698e-02 -3.55145484e-02 -3.37718520e-03  1.86522268e-02
  5.40862381e-02 -3.50071937e-02 -8.95371381e-03 -2.03199405e-02
  8.59780051e-03  7.25472122e-02 -2.56197713e-02  2.60829665e-02
  3.77264083e-03  7.42594199e-03 -1.24746151e-02 -4.97319223e-03
  4.49920855e-02 -1.05892941e-01 -6.66415840e-02  6.43658116e-02
  5.76557554e-02 -9.58824251e-03 -1.46589929e-03 -1.72939729e-02
  2.13908468e-04  6.51221140e-04  8.64628255e-02 -3.75081524e-02
 -6.25895523e-03 -7.91716352e-02 -2.95891091e-02 -5.99385351e-02
  1.15257427e-02 -4.07455

In [40]:
kb_collection = "rag_knowledge"
if not client.collection_exists(collection_name=kb_collection):
    client.create_collection(
        collection_name=kb_collection,
        vectors_config=VectorParams(size=512, distance=Distance.COSINE),
    )
client.upsert(
    collection_name=kb_collection,
    points=chunk_points
)
info = client.get_collection(collection_name=kb_collection)
rprint("Points_counts:",info.points_count)

Points_counts: 9

In [45]:
# 查询chunk
query = "地下水为什么会影响边坡稳定性？"

query_vector = model.encode(query, normalize_embeddings=True)

result = client.query_points(
    collection_name=kb_collection,
    query=query_vector.tolist(),
    limit=7,
    with_payload=True,
)

# for point in result.points:
#     rprint(f"score={point.score:.4f}")
#     rprint(point.payload)
for rank, point in enumerate(result.points, start=1):
    print(f"\nTop {rank}")
    print(f"Score: {point.score:.4f}")
    print("Source:", point.payload["source"])
    print("Chunk ID:", point.payload["chunk_id"])
    print("Text:")
    print(point.payload)


Top 1
Score: 0.8381
Source: landslide_report.pdf
Chunk ID: 2
Text:
{'text': '强度。\n\n地下水位变化同样会影响边坡稳定性。当地下水位升高时，\n坡体内部孔隙水压力可能增加，从而减小', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 2, 'document_id': 'doc_001'}

Top 2
Score: 0.7709
Source: landslide_report.pdf
Chunk ID: 1
Text:
{'text': '增加，\n并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。\n\n地下水位变化同样会影响边坡稳定', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 1, 'document_id': 'doc_001'}

Top 3
Score: 0.7303
Source: landslide_report.pdf
Chunk ID: 3
Text:
{'text': '，\n坡体内部孔隙水压力可能增加，从而减小有效应力，\n使边坡更容易发生失稳。\n\n坡度是影响滑坡发生的重', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 3, 'document_id': 'doc_001'}

Top 4
Score: 0.7197
Source: landslide_report.pdf
Chunk ID: 4
Text:
{'text': '容易发生失稳。\n\n坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，\n但滑坡发生还', 'catergory': 'la

## 8. Chunking Strategy

本节目标：

1. 理解 Fixed-size Chunking
2. 理解 Sentence-aware Chunking
3. 理解 Recursive Chunking
4. 理解 Token-based Chunking
5. 比较不同 Chunking 策略的优缺点
6. 为后续检索评测准备实验数据

In [44]:
import re

def split_sentences(text):
    # 使用正则表达式将文本切分为句子
    sentences = re.split(r'(?<=[。！？!?])', text)
    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]
    return sentences
sentences = split_sentences(document_text)

for i, sentence in enumerate(sentences):
    print(f"\nSentence {i}")
    print(sentence)


Sentence 0
持续降雨是诱发滑坡的重要因素之一。

Sentence 1
降雨入渗会导致土体含水率增加，
并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。

Sentence 2
地下水位变化同样会影响边坡稳定性。

Sentence 3
当地下水位升高时，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

Sentence 4
坡度是影响滑坡发生的重要地形条件。

Sentence 5
较大的坡度通常意味着更大的下滑力，
但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。

Sentence 6
在光伏功率预测中，太阳辐射是最重要的气象因素之一。

Sentence 7
云量、温度、湿度和风速都会影响光伏组件的发电效率。


In [46]:
def sentence_chunk_text(
    text,
    chunk_size=120
    ):
    chunks = []
    sentences = split_sentences(text)
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= chunk_size:
            current_chunk += sentence
        else:
            if current_chunk:
                chunks.append(current_chunk)
                current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk)

    return chunks
sentence_chunks = sentence_chunk_text(document_text, chunk_size=120)
print(f"chunks数量: {len(sentence_chunks)}")
for i,chunk in enumerate(sentence_chunks):
    print(f"\nchunk {i}")
    print(chunk)


chunks数量: 3

chunk 0
持续降雨是诱发滑坡的重要因素之一。降雨入渗会导致土体含水率增加，
并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。地下水位变化同样会影响边坡稳定性。

chunk 1
当地下水位升高时，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，
但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。

chunk 2
在光伏功率预测中，太阳辐射是最重要的气象因素之一。云量、温度、湿度和风速都会影响光伏组件的发电效率。


In [50]:

text = "在光伏功率预测中，太阳辐射是最重要的气象因素之一。云量、温度、湿度和风速都会影响光伏组件的发电。"
text = "in solar power prediction, solar radiation is the most important meteorological factor. Cloudiness, temperature, humidity and wind speed all affect the power generation of photovoltaic components."

tokenizer = model.tokenizer
tokens = tokenizer.encode(text, add_special_tokens=False)


print("字符数：", len(text))
print("Token数: ", len(tokens))
print(tokens)

字符数： 196
Token数:  64
[8217, 8968, 11208, 9239, 11685, 9172, 9116, 117, 8968, 11208, 12619, 9360, 8361, 8310, 8174, 11344, 10481, 9356, 9834, 8450, 8299, 8372, 8798, 10800, 9430, 12289, 9610, 119, 10279, 8620, 8565, 117, 8282, 9085, 10540, 11704, 117, 12199, 8625, 9172, 8661, 8256, 12302, 12755, 8513, 11782, 9568, 8722, 8174, 9239, 10234, 8957, 8794, 8205, 9020, 10701, 9238, 8982, 8177, 8134, 9401, 8354, 12585, 119]


In [53]:



def token_chunk_text(
        text,
        tokenizer,
        chunk_size=60,
        overlap=20
):
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + chunk_size
        chunk_token_ids = token_ids[start:end]
        chunk_text = tokenizer.decode(chunk_token_ids,skip_special_tokens=True)
        chunks.append(chunk_text)
        start += chunk_size - overlap
    return chunks

In [54]:
token_chunks = token_chunk_text(
    document_text,
    tokenizer=model.tokenizer,
    chunk_size=64,
    overlap=16
)

for i, chunk in enumerate(token_chunks):
    print(f"\nChunk {i}")
    print(chunk)


Chunk 0
持 续 降 雨 是 诱 发 滑 坡 的 重 要 因 素 之 一 。 降 雨 入 渗 会 导 致 土 体 含 水 率 增 加 ， 并 可 能 造 成 孔 隙 水 压 力 升 高 ， 从 而 降 低 土 体 有 效 应 力 和 抗 剪 强 度 。 地 下 水

Chunk 1
低 土 体 有 效 应 力 和 抗 剪 强 度 。 地 下 水 位 变 化 同 样 会 影 响 边 坡 稳 定 性 。 当 地 下 水 位 升 高 时 ， 坡 体 内 部 孔 隙 水 压 力 可 能 增 加 ， 从 而 减 小 有 效 应 力 ， 使 边

Chunk 2
可 能 增 加 ， 从 而 减 小 有 效 应 力 ， 使 边 坡 更 容 易 发 生 失 稳 。 坡 度 是 影 响 滑 坡 发 生 的 重 要 地 形 条 件 。 较 大 的 坡 度 通 常 意 味 着 更 大 的 下 滑 力 ， 但 滑 坡 发 生

Chunk 3
常 意 味 着 更 大 的 下 滑 力 ， 但 滑 坡 发 生 还 受 到 岩 性 、 土 体 结 构 、 植 被 覆 盖 和 水 文 条 件 等 因 素 共 同 控 制 。 在 光 伏 功 率 预 测 中 ， 太 阳 辐 射 是 最 重 要 的 气 象

Chunk 4
率 预 测 中 ， 太 阳 辐 射 是 最 重 要 的 气 象 因 素 之 一 。 云 量 、 温 度 、 湿 度 和 风 速 都 会 影 响 光 伏 组 件 的 发 电 效 率 。


### Chunking Strategy Comparison

| 方法 | 优点 | 缺点 |
|---|---|---|
| Fixed-size | 简单、快、长度稳定 | 容易切断语义 |
| Sentence-aware | 保留完整句子 | Chunk长度不均匀 |
| Recursive | 兼顾结构和长度 | 实现更复杂 |
| Token-based | 与模型真实上下文长度一致 | 可能仍然切断语义 |

In [56]:
test_queries = [
    {
        "query": "持续降雨为什么会增加滑坡风险？",
        "expected_keyword": "孔隙水压力"
    },
    {
        "query": "地下水位升高为什么会影响边坡？",
        "expected_keyword": "有效应力"
    },
    {
        "query": "坡度为什么会影响滑坡？",
        "expected_keyword": "下滑力"
    },
    {
        "query": "影响光伏功率的主要气象因素是什么？",
        "expected_keyword": "太阳辐射"
    }
]

In [57]:
def evaluate_retrieval(
    test_queries,
    collection_name,
    model,
    client,
    top_k=3
):
    hits = 0

    for item in test_queries:

        query = item["query"]
        expected_keyword = item["expected_keyword"]

        query_vector = model.encode(
            query,
            normalize_embeddings=True
        )

        result = client.query_points(
            collection_name=collection_name,
            query=query_vector.tolist(),
            limit=top_k
        )

        retrieved_texts = [
            point.payload["text"]
            for point in result.points
        ]

        hit = any(
            expected_keyword in text
            for text in retrieved_texts
        )

        if hit:
            hits += 1

        print("=" * 60)
        print("Query:", query)
        print("Expected:", expected_keyword)
        print("Hit:", hit)

        for point in result.points:
            print(
                f"{point.score:.4f} | "
                f"{point.payload['text']}"
            )

    hit_rate = hits / len(test_queries)

    return hit_rate

In [58]:
hit_rate = evaluate_retrieval(
    test_queries=test_queries,
    collection_name=kb_collection,
    model=model,
    client=client,
    top_k=3
)

print(
    f"\nHit Rate@3 = {hit_rate:.4f}"
)

Query: 持续降雨为什么会增加滑坡风险？
Expected: 孔隙水压力
Hit: True
0.8304 | 
持续降雨是诱发滑坡的重要因素之一。降雨入渗会导致土体含水率增加，
并可能造成孔隙水压力升高，从而降
0.7294 | ，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

坡度是影响滑坡发生的重
0.6930 | 容易发生失稳。

坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，
但滑坡发生还
Query: 地下水位升高为什么会影响边坡？
Expected: 有效应力
Hit: True
0.8458 | 强度。

地下水位变化同样会影响边坡稳定性。当地下水位升高时，
坡体内部孔隙水压力可能增加，从而减小
0.8055 | 增加，
并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。

地下水位变化同样会影响边坡稳定
0.7122 | ，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

坡度是影响滑坡发生的重
Query: 坡度为什么会影响滑坡？
Expected: 下滑力
Hit: True
0.8078 | ，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

坡度是影响滑坡发生的重
0.7990 | 容易发生失稳。

坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，
但滑坡发生还
0.7521 | 度通常意味着更大的下滑力，
但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。

在
Query: 影响光伏功率的主要气象因素是什么？
Expected: 太阳辐射
Hit: True
0.8050 | 辐射是最重要的气象因素之一。
云量、温度、湿度和风速都会影响光伏组件的发电效率。

0.7527 | 植被覆盖和水文条件等因素共同控制。

在光伏功率预测中，太阳辐射是最重要的气象因素之一。
云量、温度
0.7040 | 光伏组件的发电效率。


Hit Rate@3 = 1.0000


## 9. Retrieval Evaluation

本节目标：

1. 理解 Hit Rate@K
2. 理解 Precision@K
3. 理解 Recall@K
4. 理解 MRR
5. 构造更规范的 Ground Truth
6. 编写统一的检索评测函数
7. 为后续比较不同 Chunking / Embedding / Hybrid Search 做准备

In [77]:
# evaluation_dataset = [
#     {
#         "query": "持续降雨为什么会增加滑坡风险？",
#         "relevant_chunk_ids": [0, 1]
#     },
#     {
#         "query": "地下水位升高为什么会影响边坡稳定性？",
#         "relevant_chunk_ids": [1]
#     },
#     {
#         "query": "坡度为什么会影响滑坡发生？",
#         "relevant_chunk_ids": [2]
#     },
#     {
#         "query": "影响光伏功率的主要气象因素是什么？",
#         "relevant_chunk_ids": [3]
#     }
# ]
evaluation_dataset = [
    {
        "query": "持续降雨为什么会增加滑坡风险？",
        "relevant_chunk_ids": [0, 1]
    },
    {
        "query": "地下水位升高为什么会影响边坡稳定性？",
        "relevant_chunk_ids": [2, 3]
    },
    {
        "query": "坡度为什么会影响滑坡发生？",
        "relevant_chunk_ids": [4, 5]
    },
    {
        "query": "影响光伏功率的主要气象因素是什么？",
        "relevant_chunk_ids": [6]
    }
]

def precision_at_k(
    retrieved_ids,
    relevant_ids,
    k
):
    retrieved_top_k = retrieved_ids[:k]

    relevant_ids = set(
        relevant_ids
    )

    hits = sum(
        1
        for doc_id in retrieved_top_k
        if doc_id in relevant_ids
    )

    return hits / k

def recall_at_k(
    retrieved_ids,
    relevant_ids,
    k
):
    if len(relevant_ids) == 0:
        return 0.0

    retrieved_top_k = set(
        retrieved_ids[:k]
    )

    relevant_ids = set(
        relevant_ids
    )

    hits = len(
        retrieved_top_k & relevant_ids
    )

    return hits / len(relevant_ids)

def reciprocal_rank(
    retrieved_ids,
    relevant_ids
):
    relevant_ids = set(
        relevant_ids
    )

    for rank, doc_id in enumerate(
        retrieved_ids,
        start=1
    ):
        if doc_id in relevant_ids:
            return 1 / rank

    return 0.0

def hit_rate_at_k(
    retrieved_ids,
    relevant_ids,
    k
):
    retrieved_top_k = set(
        retrieved_ids[:k]
    )

    relevant_ids = set(
        relevant_ids
    )

    hit = len(
        retrieved_top_k & relevant_ids
    ) > 0

    return int(hit)

In [60]:
import numpy as np

#统一的评测函数，方便后续比较不同策略hit_rare@k, precision@k, recall@k, mrr
def evaluate_retriever(
    evaluation_dataset,
    collection_name,
    model,
    client,
    top_k=3
):
    hit_scores = []
    precision_scores = []
    recall_scores = []
    rr_scores = []

    details = []

    for item in evaluation_dataset:

        query = item["query"]

        relevant_ids = item[
            "relevant_chunk_ids"
        ]

        query_vector = model.encode(
            query,
            normalize_embeddings=True
        )

        result = client.query_points(
            collection_name=collection_name,
            query=query_vector.tolist(),
            limit=top_k
        )

        retrieved_ids = [
            point.id
            for point in result.points
        ]

        hit = hit_rate_at_k(
            retrieved_ids,
            relevant_ids,
            top_k
        )

        precision = precision_at_k(
            retrieved_ids,
            relevant_ids,
            top_k
        )

        recall = recall_at_k(
            retrieved_ids,
            relevant_ids,
            top_k
        )

        rr = reciprocal_rank(
            retrieved_ids,
            relevant_ids
        )

        hit_scores.append(hit)
        precision_scores.append(
            precision
        )
        recall_scores.append(
            recall
        )
        rr_scores.append(rr)

        details.append({
            "query": query,
            "retrieved_ids": retrieved_ids,
            "relevant_ids": relevant_ids,
            "hit": hit,
            "precision": precision,
            "recall": recall,
            "rr": rr
        })

    metrics = {
        f"HitRate@{top_k}":
            np.mean(hit_scores),

        f"Precision@{top_k}":
            np.mean(precision_scores),

        f"Recall@{top_k}":
            np.mean(recall_scores),

        "MRR":
            np.mean(rr_scores)
    }

    return metrics, details

In [78]:
metrics, details = evaluate_retriever(
    evaluation_dataset=evaluation_dataset,
    collection_name=kb_collection,
    model=model,
    client=client,
    top_k=3
)
metrics

{'HitRate@3': np.float64(1.0),
 'Precision@3': np.float64(0.49999999999999994),
 'Recall@3': np.float64(0.875),
 'MRR': np.float64(1.0)}

In [76]:
from qdrant_client.models import Distance, PointStruct, VectorParams

if client.collection_exists(collection_name=kb_collection):
    client.delete_collection(collection_name=kb_collection)

client.create_collection(
    collection_name=kb_collection,
    vectors_config=VectorParams(
        size=512,
        distance=Distance.COSINE
    )
)
print("collection已重建", kb_collection)
# print(chunks)
document_texts = """
持续降雨是诱发滑坡的重要因素之一。降雨入渗会导致土体含水率增加，
并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。

地下水位变化同样会影响边坡稳定性。当地下水位升高时，
坡体内部孔隙水压力可能增加，从而减小有效应力，
使边坡更容易发生失稳。

坡度是影响滑坡发生的重要地形条件。较大的坡度通常意味着更大的下滑力，
但滑坡发生还受到岩性、土体结构、植被覆盖和水文条件等因素共同控制。

在光伏功率预测中，太阳辐射是最重要的气象因素之一。
云量、温度、湿度和风速都会影响光伏组件的发电效率。
"""
# sentences = split_sentences(document_texts)
# rprint(sentences)
# 分chunks
chunks = sentence_chunk_text(document_texts,chunk_size=60)

#构建metadata

chunk_records = []
for i, chunk in enumerate(chunks):
    chunk_record = {
        "text": chunk,
        "catergory": "landslide",
        "source": "landslide_report.pdf",
        "year": 2026,
        "page": 12,
        "region": "Kentucky",
        "chunk_id": i,
        "document_id": "doc_001"
    }
    chunk_records.append(chunk_record)
print(chunk_records[0])
#批量embedding
chunk_texts = [record["text"] for record in chunk_records]
chunk_embedding = model.encode(chunk_texts, normalize_embeddings=True,show_progress_bar=True)
# print(chunk_embedding[0])
# 批量Upsert到Qdrant
chunk_points = []
for i, (chunk_text, embedding) in enumerate(zip(chunk_records, chunk_embedding)):
    point = PointStruct(
        id=i, #  点的ID，用于标识唯一向量
        vector=embedding.tolist(), #  将编码后的向量转换为列表格式
        payload=chunk_text
    )
    chunk_points.append(point)
for point in chunk_points:
    print(point.payload)
# print(chunk_points[0]) 
# 重建collection后，重新插入数据
client.upsert(
    collection_name=kb_collection,
    points=chunk_points
    
)
info = client.get_collection(collection_name=kb_collection)
print(info.points_count)

collection已重建 rag_knowledge
{'text': '持续降雨是诱发滑坡的重要因素之一。', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 0, 'document_id': 'doc_001'}


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.54it/s]

{'text': '持续降雨是诱发滑坡的重要因素之一。', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 0, 'document_id': 'doc_001'}
{'text': '降雨入渗会导致土体含水率增加，\n并可能造成孔隙水压力升高，从而降低土体有效应力和抗剪强度。', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 1, 'document_id': 'doc_001'}
{'text': '地下水位变化同样会影响边坡稳定性。', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 2, 'document_id': 'doc_001'}
{'text': '当地下水位升高时，\n坡体内部孔隙水压力可能增加，从而减小有效应力，\n使边坡更容易发生失稳。', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 3, 'document_id': 'doc_001'}
{'text': '坡度是影响滑坡发生的重要地形条件。', 'catergory': 'landslide', 'source': 'landslide_report.pdf', 'year': 2026, 'page': 12, 'region': 'Kentucky', 'chunk_id': 4, 'document_id': 'doc_001'}
{'text': '较大的坡度通常意味着更大的下滑力，\n但滑坡发生还受到岩性、土体结构、植

In [79]:
metrics, details = evaluate_retriever(
    evaluation_dataset=evaluation_dataset,
    collection_name=kb_collection,
    model=model,
    client=client,
    top_k=3
)
metrics

{'HitRate@3': np.float64(1.0),
 'Precision@3': np.float64(0.49999999999999994),
 'Recall@3': np.float64(0.875),
 'MRR': np.float64(1.0)}